In [ ]:
from pathlib import Path

from pydantic import BaseModel, ConfigDict
from typing import cast
import pickle as pkl

from skimage.graph import route_through_array
import matplotlib.pyplot as plt
import numpy.typing as npt
import numpy as np

import networkx as nx

from topostats.classes import TopoStats
from topostats.plottingfuncs import Colormap

from skimage.measure import label

from topostats.utils import convolve_skeleton

CMAP = Colormap().get_cmap()
VMIN = -3
VMAX = 4

In [ ]:
def plot(image: npt.NDArray[np.uint8]) -> None:
    plt.imshow(image, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    plt.show()


def create_mask_from_points(image_shape: tuple[int, int], points: list[tuple[int, int]]) -> npt.NDArray[np.uint8]:
    mask = np.zeros(image_shape, dtype=np.uint8)
    for point in points:
        mask[point[0], point[1]] = 1
    return mask


def plot_mask_of_points(image_shape: tuple[int, int], points: list[tuple[int, int]]) -> None:
    mask = create_mask_from_points(image_shape, points)
    plt.imshow(mask, cmap="gray")
    plt.show()

In [ ]:
from typing import Generator
import itertools
from typing import TypeVar


def get_permutations_of_options_with_costs(
    dict_of_items_with_options_and_costs: dict[int, list[tuple[int, float]]],
) -> list[tuple[dict[int, tuple[int, float]], float]]:
    """
    Get costs of all permutations of options for a dictionary of items with varying options.

    Given a dictionary of items with their representative options and costs, return all permutations of the options
    with their associated total costs.

    Parameters
    ----------
    dict_of_items_with_options_and_costs : dict[int, list[tuple[int, float]]]
        A dictionary where keys are item identifiers and values are lists of tuples, each containing an option and its
        cost.


    Returns
    -------
    list[tuple[dict[int, tuple[int, float]], float]]
        A list of tuples, where each tuple contains a dictionary mapping item identifiers to their selected option and
        the total cost of that permutation.
    """
    # Unpack the items (ID, options) to compute all combinations
    keys, option_lists = zip(*dict_of_items_with_options_and_costs.items())
    all_permutations = []
    for combination in itertools.product(*option_lists):
        permutation_dict = dict(zip(keys, combination))
        total_cost = sum(cost for _, cost in combination)
        all_permutations.append((permutation_dict, total_cost))
    return all_permutations


T = TypeVar("T")


def get_all_pairing_lists_for_items(items: list[T]) -> Generator[list[tuple[T, T]], None, None]:
    """
    Generates all possible pairing lists for a given list of items.

    Parameters
    ----------
    items : list[T]
        A list of items to be paired.

    Yields
    ------
    list[tuple[T, T]]
        A list of tuples, where each tuple contains a pair of items.
    """
    # Base case: no items left
    if not items:
        yield []
        return

    # Choose the current item to pair with the rest
    first = items[0]
    rest = items[1:]

    # Pair this item with each of the remaining items
    for i, partner in enumerate(rest):
        pair = (first, partner)
        remaining_items = rest[:i] + rest[i + 1 :]

        # Recursively pair the remaining elements that we don't pair with the current item
        for sub_pairing in get_all_pairing_lists_for_items(remaining_items):
            yield [pair] + sub_pairing


def create_vertex_edge_to_edge_map(
    vertex_suggestions: dict[int, list[tuple[int, int]]],
) -> dict[tuple[int, int], int]:
    """Create a mapping of vertex -> in_edge, out_edge based on the suggestions for which branches to pair."""
    # Create a mapping of vertex -> in_edge, out_edge
    vertex_edge_to_edge_map: dict[tuple[int, int], int] = {}  # (vertex_index, in_edge_index) -> out_edge_index
    for vertex_index, suggestions in vertex_suggestions.items():
        for in_edge_index, out_edge_index in suggestions:
            vertex_edge_to_edge_map[(vertex_index, in_edge_index)] = out_edge_index
            vertex_edge_to_edge_map[(vertex_index, out_edge_index)] = in_edge_index

    return vertex_edge_to_edge_map


def get_cycles_from_graph(
    edges_to_vertices: dict[int, tuple[int, int]], vertex_suggestions: dict[int, list[tuple[int, int]]]
) -> list[list[tuple[int, int]]]:
    """
    Get cycles from a 2k-regular graph, using suggestions for which branches to pair.
    """
    vertex_edge_to_edge_map = create_vertex_edge_to_edge_map(vertex_suggestions)

    # Construct cycles based on the vertex_edge_to_edge_map
    cycles: list[list[tuple[int, int]]] = []  # Each cycle is a list of (edge_index, vertex_index) tuples
    current_cycle: list[tuple[int, int]] = []
    # Start at a vertex, choose an edge and follow it
    start_vertex = next(iter(vertex_suggestions.keys()))
    current_vertex = start_vertex
    current_vertex_suggestions = vertex_suggestions[current_vertex]
    current_vertex_suggestion = current_vertex_suggestions[0]
    current_edge = current_vertex_suggestion[0]
    visited_edges = set()
    while True:
        # Add the current edge to the cycle
        current_cycle.append((current_edge, current_vertex))
        # Get the next edge to follow
        next_edge = vertex_edge_to_edge_map.get((current_vertex, current_edge))
        # Get the next vertex to move to
        edge_vertices = edges_to_vertices[current_edge]
        # Avoid staying on the same vertex
        next_vertex = edge_vertices[0] if edge_vertices[1] == current_vertex else edge_vertices[1]
        # Check if we've visited this edge before
        if next_edge in visited_edges:
            # Completed a cycle
            cycles.append(current_cycle)
            # Start a new cycle if there are unvisited edges
            unvisited_edges = set(edges_to_vertices.keys()) - visited_edges
            if not unvisited_edges:
                break  # All edges visited, exit the loop
            # Start a new cycle with an unvisited edge
            current_edge = unvisited_edges.pop()
            current_vertex = edges_to_vertices[current_edge][0]  # Start from one of the vertices of the new edge
            current_cycle = []
            continue
        else:
            # Not completed a cycle, mark the current edge as visited and move to the next vertex
            visited_edges.add(current_edge)
            current_edge = next_edge
            current_vertex = next_vertex

    assert len(visited_edges) == len(edges_to_vertices), "Not all edges were visited in the traversal"

    return cycles


def get_cycles_from_endpoint_node_collection(
    endpoint_node_collection: dict[str, int],
    endpoint_endpoint_vertex_connection_suggestions: dict[int, list[tuple[str, str]]],
) -> list[list[tuple[str, int]]]:
    """
    Get cycles from a collection of endpoints and the associated node.

    Endpoint ids are of form "int_int" where the first int is the branch id, there are always two endpoints per
    branch so endpoint 3_0 connects to endpoint 3_1 and each is connected to a node.
    """
    pass


def create_single_trace(
    vertex_suggestions_with_costs: dict[int, list[tuple[Endpoint, Endpoint, float]]],
) -> tuple[list[Endpoint], float]:
    """
    Create a single Eulerian trace from a 2k-regular graph, using suggestions for which branches to pair.

    Parameters
    ----------
    edges_to_vertices : dict
        A dictionary mapping edge indexes to two vertex indexes.
    vertex_suggestions : dict
        A dictionary mapping vertex indexes to a list of tuples, each containing two branch indexes and a
        confidence score.

    Returns
    -------
    tuple[list[Endpoint], float]
        A list of Endpoint to Endpoint connections representing a single Eulerian trace, and the confidence cost
        associated with the whole trace.
    """
    print(f"vertex_suggestions_with_costs: {vertex_suggestions_with_costs}")
    all_endpoint_ids: set[str] = set()
    for suggestions in vertex_suggestions_with_costs.values():
        for in_endpoint, out_endpoint, confidence in suggestions:
            all_endpoint_ids.add(in_endpoint.id)
            all_endpoint_ids.add(out_endpoint.id)
    print(f"all_endpoint_ids: {all_endpoint_ids}")
    # For each vertex, get the permutations of the suggestions with associated costs
    # Dict of vertex_index -> list of pairing options with the total cost of that pairing option
    vertex_permutations_with_costs: dict[int, list[tuple[list[tuple[str, str]], float]]] = {}
    # Iterate over the vertexes and their suggestions to create pairing options and calculate costs
    for vertex_index, suggestions in vertex_suggestions_with_costs.items():
        all_endpoint_indexes: set[str] = set()
        for in_endpoint, out_endpoint, confidence in suggestions:
            all_endpoint_indexes.add(in_endpoint.id)
            all_endpoint_indexes.add(out_endpoint.id)
        # Get all permutations of the possible connections for this vertex
        all_pairing_options = list(get_all_pairing_lists_for_items(list(all_endpoint_indexes)))
        print(
            f"all pairing options for vertex {vertex_index} | all endpoints: {all_endpoint_indexes}: {all_pairing_options}"
        )
        # Associate costs with these pairings
        all_pairing_options_with_costs: list[tuple[list[tuple[str, str]], float]] = (
            []
        )  # list of (pairing_option, cost) tuples
        for pairing_option in all_pairing_options:
            cost = 0
            for in_endpoint, out_endpoint, confidence in suggestions:
                in_endpoint_id = in_endpoint.id
                out_endpoint_id = out_endpoint.id
                if (in_endpoint_id, out_endpoint_id) not in pairing_option and (
                    out_endpoint_id,
                    in_endpoint_id,
                ) not in pairing_option:
                    cost += confidence  # Add the confidence score to the cost if this suggestion is not used
            all_pairing_options_with_costs.append((pairing_option, cost))
            print(f"| pairing option: {pairing_option} | cost: {cost}")
        vertex_permutations_with_costs[vertex_index] = all_pairing_options_with_costs

    for vertex_index, pairing_options_with_costs in vertex_permutations_with_costs.items():
        print(f"vertex {vertex_index}")
        for pairing_option, cost in pairing_options_with_costs:
            print(f"| pairing option: {pairing_option} | cost: {cost}")

    # So here we have a list of possible pairing options for each vertex with their costs.

    # Now want to permute these globally across all vertices

    print("==== global permutations ====")

    # Create a simple mapping between vertex index and the pairing options with costs for that vertex - just
    # for simplicity with handling the itertools product later on
    vertex_pairing_options_with_costs: dict[int, list[tuple[int, float]]] = (
        {}
    )  # vertex_index -> list of (pairing_option_index, cost) tuples
    for vertex_index, pairing_options_with_costs in vertex_permutations_with_costs.items():
        vertex_pairing_options_with_costs[vertex_index] = []
        # For each pairing option with cost, add that
        for pairing_option_index, (_pairing_option, cost) in enumerate(pairing_options_with_costs):
            vertex_pairing_options_with_costs[vertex_index].append((pairing_option_index, cost))

    # print for debugging
    # for vertex_index, pairing_options_with_costs in vertex_pairing_options_with_costs.items():
    #     print(f"vertex {vertex_index}")
    #     for pairing_option_index, cost in pairing_options_with_costs:
    #         print(f"| pairing option index: {pairing_option_index} | cost: {cost}")

    # Find the combination of pairing options across all vertices and their costs
    # list of ({vertex_index: (pairing_option_index, cost)}, total_cost) tuples
    global_all_vertex_pairing_options_with_costs = get_permutations_of_options_with_costs(
        vertex_pairing_options_with_costs
    )

    # Sort the global pairing options by their total cost in ascending order
    global_all_vertex_pairing_options_with_costs.sort(key=lambda x: x[1])

    # print for debugging
    # for global_pairing_option, total_cost in global_all_vertex_pairing_options_with_costs:
    #     print(f"global pairing option: {global_pairing_option} | total cost: {total_cost}")
    #     for vertex_index, (pairing_option_index, cost) in global_pairing_option.items():
    #         print(f"| vertex {vertex_index} | pairing option index: {pairing_option_index} | cost: {cost}")

    print("=== global pairing options with costs reconstruction ===")

    # Iterate through the sorted global pairing options and find the first one that results in a single Eulerian trace

    found_single_cycle = False
    for global_pairing_option, total_cost in global_all_vertex_pairing_options_with_costs:
        print(f"global pairing option: {global_pairing_option} | total cost: {total_cost}")
        # global pairing option is a dict of vertex_index -> (pairing_option_index, cost)
        # for vertex_index, (pairing_option_index, cost) in global_pairing_option.items():
        #     # Get the pairing option for this vertex
        #     pairing_option = vertex_permutations_with_costs[vertex_index][pairing_option_index]
        #     print(f"pairing option: {pairing_option} | cost: {cost}")
        # Create cycles
        cycles: list[list[str]] = []
        current_cycle: list[str] = []
        # Start at an endpoint, any endpoint - grab the first one from the endpoint set
        first_endpoint_id = next(iter(all_endpoint_ids))
        # Now follow the first endpoint through the global pairing option to create a cycle
        visited_endpoint_ids: set[str] = set()
        current_branch_start_endpoint_id = first_endpoint_id
        print(f"finding cycles:")

        while True:
            # Check if we have already visited this endpoint, if so we have completed a cycle

            if current_branch_start_endpoint_id in visited_endpoint_ids:
                print(f"  completed a cycle with endpoint {current_branch_start_endpoint_id}")
                # completed a cycle - don't add the current endpoint to the cycle since it is already in there
                cycles.append(current_cycle)
                unvisited_endpoint_ids = all_endpoint_ids - visited_endpoint_ids
                print(f"  unvisited_endpoint_ids: {unvisited_endpoint_ids}")
                # if all endpoints have been visited, break
                if len(unvisited_endpoint_ids) == 0:
                    print(f"  all endpoints have been visited, breaking")
                    break
                # Start a new cycle with an unvisited endpoint
                current_branch_start_endpoint_id = unvisited_endpoint_ids.pop()
                current_cycle = []
                print(f"  starting new cycle with endpoint {current_branch_start_endpoint_id}")

            print(f"  current branch_start_endpoint_id: {current_branch_start_endpoint_id}")
            # Add the start endpoint
            visited_endpoint_ids.add(current_branch_start_endpoint_id)
            current_cycle.append(current_branch_start_endpoint_id)
            # Get the corresponding endpoint, ie 5_0 if current is 5_1 or vice versa
            if current_branch_start_endpoint_id.endswith("_0"):
                current_branch_end_endpoint_id = current_branch_start_endpoint_id.replace("_0", "_1")
            elif current_branch_start_endpoint_id.endswith("_1"):
                current_branch_end_endpoint_id = current_branch_start_endpoint_id.replace("_1", "_0")
            else:
                raise ValueError(f"Invalid endpoint id: {current_branch_start_endpoint_id}")
            print(f"  current branch_end_endpoint_id: {current_branch_end_endpoint_id}")

            # Add the end endpoint
            visited_endpoint_ids.add(current_branch_end_endpoint_id)
            current_cycle.append(current_branch_end_endpoint_id)

            print(f"  current cycle: {current_cycle}")

            # Find out the next start endpoint to follow
            # Get the node index for this endpoint
            current_endpoint = next(endpoint for endpoint in endpoints if endpoint.id == current_branch_end_endpoint_id)
            current_node_index = current_endpoint.node_index
            print(f"  current_node_index: {current_node_index}")
            # Get the pairing option for this node
            node_pairing_option_index, _cost = global_pairing_option[current_node_index]
            pairing_option = vertex_permutations_with_costs[current_node_index][node_pairing_option_index][
                0
            ]  # [0] to get the pairing option from the (pairing_option, cost) tuple
            print(f" pairing_option: {pairing_option} | node_pairing_option_index: {node_pairing_option_index}")

            # Get the corresponding endpoint for this endpoint in the pairing option
            next_branch_start_endpoint_id = None
            for endpoint_id_0, endpoint_id_1 in pairing_option:
                print(f"  checking pair: ({endpoint_id_0}, {endpoint_id_1})")
                if endpoint_id_0 == current_branch_end_endpoint_id:
                    # Contains, and therefore choose the other
                    next_branch_start_endpoint_id = endpoint_id_1
                    break
                elif endpoint_id_1 == current_branch_end_endpoint_id:
                    # Contains, and therefore choose the other
                    next_branch_start_endpoint_id = endpoint_id_0
                    break
            assert (
                next_branch_start_endpoint_id is not None
            ), f"Could not find a pair for endpoint {current_branch_end_endpoint_id}"
            print(f"  next_branch_start_endpoint_id: {next_branch_start_endpoint_id}")

            current_branch_start_endpoint_id = next_branch_start_endpoint_id

        print(f"----")
        if len(cycles) == 1:
            print(
                f"Found a single cycle with global pairing option: {global_pairing_option} | total cost: {total_cost}"
            )
            found_single_cycle = True
            break
        else:
            print(
                f"Found {len(cycles)} cycles with global pairing option: {global_pairing_option} | total cost: {total_cost}"
            )
            # Try the next one

    assert (
        found_single_cycle
    ), "Could not find a single cycle with any of the global pairing options - this should never happen"
    print(f"cycles: {cycles}")

    # Construct a list of Endpoint objects for the trace, rather than just indexes
    trace: list[Endpoint] = []
    single_cycle = cycles[0]
    # Iterate over the cycle and create a list of Endpoint objects for the trace
    for endpoint_id in single_cycle:
        endpoint = next(endpoint for endpoint in endpoints if endpoint.id == endpoint_id)
        trace.append(endpoint)

    return trace, total_cost

In [ ]:
with open(Path("../debug_topostats.pkl"), "rb") as f:
    topostats = cast(TopoStats, pkl.load(f))
assert isinstance(topostats, TopoStats), "Loaded object is not a TopoStats instance"
assert topostats is not None


class Endpoint(BaseModel):
    id: str
    branch_id: int
    coords: tuple[int, int]
    node_index: int

    @classmethod
    def validate_branch_id(cls, v):
        if not isinstance(v, int) or v < 2:
            raise ValueError("branch_id must be an integer greater than 1")
        return v

    def require_node_index(self) -> int:
        if self.node_index is None:
            raise ValueError(f"Endpoint {self.id} does not have a node_index assigned")
        return self.node_index

    # string repr
    def __str__(self) -> str:
        return f"Endpoint(id={self.id}, branch_id={self.branch_id}, coords={self.coords}, node_index={self.node_index})"


print(topostats)
print(topostats.grain_crops)
grain_crops = topostats.require_grain_crops()
grain_crop = grain_crops[2]
plot(grain_crop.image)
nodes = grain_crop.nodes
print(nodes.keys())
node_0 = nodes[0]
node_1 = nodes[1]
print(node_0)
print(node_0.branch_stats.keys())
print(node_0.unmatched_branch_stats.keys())


plt.imshow(grain_crop.image)
plt.imshow(node_0.node_branch_mask, alpha=0.5)
print("=== branch stats ===")
for branch_id, branch_stat in node_0.branch_stats.items():
    print(f"---branch id {branch_id}:\n{branch_stat}")
    random_color = np.random.rand(
        3,
    )
    for coord in branch_stat.ordered_coords:
        plt.plot(coord[1], coord[0], color=random_color, marker="o", markersize=2)

for coord in node_0.node_coords:
    plt.plot(coord[1], coord[0], "ro", markersize=3)
plt.show()

# Create a skeleton image without the nodes - just branches
grain_branches_image = grain_crop.skeleton
plt.imshow(grain_branches_image)
plt.show()
for node_index, node in nodes.items():
    print(f"=== node index {node_index} ===")
    node_coords = node.node_coords
    assert node_coords is not None, "Node coordinates are None"
    for node_coord in node_coords:
        grain_branches_image[node_coord[0], node_coord[1]] = -1
plt.imshow(grain_branches_image)
plt.show()

# Label the branches
grain_branches_image_without_nodes = np.where(grain_branches_image == -1, 0, grain_branches_image)
endpoint_mask = convolve_skeleton(grain_branches_image_without_nodes) == 2
plt.imshow(endpoint_mask)
plt.show()
plt.imshow(grain_branches_image_without_nodes)
plt.show()
labelled_grain_branches_image = label(grain_branches_image_without_nodes)
# add the node pixels back to the labelled image
labelled_branches_image = np.where(grain_branches_image == -1, -1, labelled_grain_branches_image)
plt.imshow(labelled_branches_image)
plt.show()


endpoints: list[Endpoint] = []
# For each branch, look at each end and see what node it is touching, then associate that branch with that node
for branch_id in np.unique(labelled_branches_image):
    if branch_id == 0 or branch_id == -1:  # Skip background and node pixels
        continue
    # Find the endpoints of the branch by convolving the branch
    branch_mask = np.where(labelled_branches_image == branch_id, 1, 0)
    convolved_branch = convolve_skeleton(branch_mask)
    plt.imshow(convolved_branch)
    plt.show()
    endpoints_coords = np.argwhere(convolved_branch == 2)
    assert len(endpoints_coords) == 2, f"Branch {branch_id} does not have exactly 2 endpoints: {endpoints_coords}"
    # Find which nodes the endpoints are touching
    for endpoint_index, endpoint_coord in enumerate(endpoints_coords):
        node_indices: list[int] = []
        node_index = None
        endpoint_id = f"{branch_id}_{endpoint_index}"
        # Get the nodes connected to this endpoint
        for node_index, node in nodes.items():
            node_confidence = node.confidence
            assert node_confidence is not None, "Node confidence is None"
            assert node.node_coords is not None, "Node coordinates are None"
            endpoint_neighbour_pixels = [
                (endpoint_coord[0] + dy, endpoint_coord[1] + dx) for dy in [-1, 0, 1] for dx in [-1, 0, 1]
            ]
            if any(
                endpoint_neighbour_pixel in node.node_coords for endpoint_neighbour_pixel in endpoint_neighbour_pixels
            ):
                node_indices.append(node_index)
                break
        assert (
            len(node_indices) == 1
        ), f"endpoint {endpoint_id} of branch {branch_id} is not connected to exactly 1 node: {node_indices}"
        # endpoint_node_connections[Endpoint(id=endpoint_id, branch_id=branch_id, coords=(endpoint[0], endpoint[1]))] = (
        # node_indices[0]
        # )
        endpoints.append(
            Endpoint(
                id=endpoint_id,
                branch_id=branch_id,
                coords=(endpoint_coord[0], endpoint_coord[1]),
                node_index=node_indices[0],
            )
        )


# print(f"=== endpoint_node_connections ===\n{endpoint_node_connections}")

# Construct a suggestions for each node for which branches to pair together
# branch_connection_suggestions: dict[int, list[tuple[int, int, float]]] = (
# {}
# )  # node_index -> (branch_index_1, branch_index_2, confidence)
endpoint_connection_suggestions: dict[int, list[tuple[Endpoint, Endpoint, float]]] = (
    {}
)  # node_index -> (endpoint_id_1, endpoint_id_2, confidence)
for node_index, node in nodes.items():
    node_pairing_suggestions: list[tuple[Endpoint, Endpoint, float]] = []
    node_confidence = node.confidence
    assert node_confidence is not None
    node_branch_stats = node.branch_stats
    assert node_branch_stats is not None
    for node_branch_pair_data in node_branch_stats.values():
        branch_pair_coords = node_branch_pair_data.ordered_coords
        assert branch_pair_coords is not None
        # Get the indexes of the branches in the pair
        endpoint_pair: list[Endpoint] = []
        for branch_pair_coord in branch_pair_coords:
            # skip if the coord cooresponds to a node pixel
            if labelled_branches_image[branch_pair_coord[0], branch_pair_coord[1]] == -1:
                continue
            # check if an endpoint
            for endpoint in endpoints:
                if endpoint.coords == (branch_pair_coord[0], branch_pair_coord[1]):
                    endpoint_pair.append(endpoint)
                    break
        assert (
            len(endpoint_pair) == 2
        ), f"Branch pair coordinates {branch_pair_coords} do not correspond to exactly 2 endpoints: {endpoint_pair}"
        node_pairing_suggestions.append((endpoint_pair[0], endpoint_pair[1], float(np.round(node_confidence, 3))))
    endpoint_connection_suggestions[node_index] = node_pairing_suggestions

print(f"=== endpoint_connection_suggestions ===\n{endpoint_connection_suggestions}")


endpoint_connections, cost = create_single_trace(vertex_suggestions_with_costs=endpoint_connection_suggestions)

print(endpoint_connections)

# Compile the trace.
trace: list[tuple[int, int]] = []
initial_endpoint = endpoint_connections[0]
branch_start_endpoint = endpoint_connections.pop(0)
final_endpoint = False

while True:
    # Add the current endpoint to the trace
    trace.append((branch_start_endpoint.coords[0], branch_start_endpoint.coords[1]))
    branch_end_endpoint = endpoint_connections.pop(0)
    # Check that the two endpoints are of the same branch
    assert (
        branch_start_endpoint.branch_id == branch_end_endpoint.branch_id
    ), f"Endpoints {branch_start_endpoint} and {branch_end_endpoint} are not of the same branch"

    current_branch_id = branch_start_endpoint.branch_id

    # Trace the branch from current endpoint to next endpoint
    branch_mask = np.where(labelled_branches_image == current_branch_id, 1, 0)
    branch_start_endpoint_coords = branch_start_endpoint.coords
    branch_end_endpoint_coords = branch_end_endpoint.coords
    # Get the coordinates of the branch pixels
    disordered_branch_coords = np.argwhere(branch_mask == 1)
    # Order the branch coords by pathfinding from start to end endpoint
    branch_pathfinding_weights = np.invert(
        branch_mask.astype(bool)
    )  # Invert the branch mask to create weights for pathfinding
    branch_pathfinding_weights = np.where(
        branch_pathfinding_weights == 0, 1, 1000000
    )  # Set weights: 1 for branch pixels, high for others
    branch_route, cost = route_through_array(
        branch_pathfinding_weights, branch_start_endpoint_coords, branch_end_endpoint_coords
    )
    # Check if the branch starts or ends with the current endpoint, if it ends with it, flip it
    branch_coords = np.array(branch_route)
    branch_start_coords = branch_coords[0]
    branch_end_coords = branch_coords[-1]
    if (
        branch_end_coords[0] == branch_start_endpoint_coords[0]
        and branch_end_coords[1] == branch_start_endpoint_coords[1]
    ):
        branch_coords = np.flip(branch_coords, axis=0)

    # Add the branch coordinates to the trace
    for branch_coord in branch_coords:
        trace.append((branch_coord[0], branch_coord[1]))

    # Add the branch end endpoint to the trace
    trace.append((branch_end_endpoint_coords[0], branch_end_endpoint_coords[1]))

    # Get the node index for the next endpoint
    connected_node_index = branch_end_endpoint.node_index

    # Get the next endpoint to connect to
    if len(endpoint_connections) == 0:
        # This is the final endpoint to traverse, so we get the initial endpoint instead
        next_branch_start_endpoint = initial_endpoint
        final_endpoint = True
    else:
        # This is not the final endpoint to traverse, so get the next endpoint to connect to from the
        # list of endpoint connections
        next_branch_start_endpoint = endpoint_connections.pop(0)

    # Trace through the node to the next endpoint
    node_mask = np.zeros_like(labelled_branches_image)
    node = nodes[connected_node_index]
    assert node.node_coords is not None, "Node coordinates are None"
    for node_coord in node.node_coords:
        node_mask[node_coord[0], node_coord[1]] = 1

    # Trace the node from the current endpoint to the next endpoint
    pathfinding_weights = np.invert(node_mask.astype(bool))  # Invert the node mask to create weights for pathfinding
    pathfinding_weights = np.where(
        pathfinding_weights == 0, 1, 1000000
    )  # Set weights: 1 for node pixels, high for others
    route = route_through_array(pathfinding_weights, branch_end_endpoint_coords, next_branch_start_endpoint.coords)

    # Add the route coordinates to the trace
    for route_coord in route[0]:
        trace.append((route_coord[0], route_coord[1]))

    if final_endpoint:
        # We are done, having traversed through a node to reach the initial endpoint again
        break

    # Set the next branch start endpoint as the current branch start endpoint for the next iteration
    branch_start_endpoint = next_branch_start_endpoint

# Construct an image of the resulting trace with increasing pixel values along the trace for visualisation
pixel_trace_image = np.zeros_like(labelled_branches_image)
height = 0
for trace_coord in trace:
    pixel_trace_image[trace_coord[0], trace_coord[1]] = height
    height += 1

plt.imshow(grain_crop.image, cmap=CMAP, vmin=VMIN, vmax=VMAX)
plt.imshow(pixel_trace_image, alpha=0.5)
plt.show()

In [ ]:
from typing import Generator


def get_all_pairing_lists_for_items(items: list[int]) -> Generator[list[tuple[int, int]], None, None]:
    """
    Generates all possible pairing lists for a given list of items.

    Parameters
    ----------
    items : list[int]
        A list of items to be paired.

    Yields
    ------
    list[tuple[int, int]]
        A list of tuples, where each tuple contains a pair of items.
    """
    # Base case: no items left
    if not items:
        yield []
        return

    # Choose the current item to pair with the rest
    first = items[0]
    rest = items[1:]

    # Pair this item with each of the remaining items
    for i, partner in enumerate(rest):
        pair = (first, partner)
        remaining_items = rest[:i] + rest[i + 1 :]

        # Recursively pair the remaining elements that we don't pair with the current item
        for sub_pairing in get_all_pairing_lists_for_items(remaining_items):
            yield [pair] + sub_pairing


# Example with 6 items
items = ["A", "B", "C", "D", "E", "F"]
pairings = list(get_all_pairing_lists_for_items(items))

print(f"Total pairings for {len(items)} items: {len(pairings)}")
for pair in pairings:
    print(pair)

absent_pair_costs = {
    ("A", "B"): 0.1,
    ("C", "D"): 0.2,
    ("E", "F"): 0.3,
}

# Add the costs if the pair is not present in the list
pairings_with_costs = []
for pairing_list in pairings:
    cost = 0
    for absent_pair, cost_value in absent_pair_costs.items():
        if absent_pair not in pairing_list:
            cost += cost_value
    pairings_with_costs.append((pairing_list, cost))
print(f"Total pairings with costs for {len(items)} items: {len(pairings_with_costs)}")
for p, c in pairings_with_costs:
    print(f"Pairing: {p}, Cost: {c}")

In [ ]:
import itertools

# Key-value mapping of list IDs to their respective options
vertex_options_and_costs: dict[int, list[tuple[int, float]]] = {
    0: [(0, 0.0), (1, 0.2)],
    1: [(0, 0.0), (1, 0.4)],
}

# Unpack the items (ID, options) to compute all combinations
keys, option_lists = zip(*vertex_options_and_costs.items())
all_vertex_pairing_options = [dict(zip(keys, combination)) for combination in itertools.product(*option_lists)]

print(f"Total vertex pairing options: {len(all_vertex_pairing_options)}")
for option in all_vertex_pairing_options:
    print(option)


def get_permutations_of_options_with_costs(
    dict_of_items_with_options_and_costs: dict[int, list[tuple[int, float]]],
) -> list[tuple[dict[int, tuple[int, float]], float]]:
    """
    Get costs of all permutations of options for a dictionary of items with varying options.

    Given a dictionary of items with their representative options and costs, return all permutations of the options
    with their associated total costs.

    Parameters
    ----------
    dict_of_items_with_options_and_costs : dict[int, list[tuple[int, float]]]
        A dictionary where keys are item identifiers and values are lists of tuples, each containing an option and its
        cost.


    Returns
    -------
    list[tuple[dict[int, tuple[int, float]], float]]
        A list of tuples, where each tuple contains a dictionary mapping item identifiers to their selected option and
        the total cost of that permutation.
    """
    # Unpack the items (ID, options) to compute all combinations
    keys, option_lists = zip(*dict_of_items_with_options_and_costs.items())
    all_permutations = []
    for combination in itertools.product(*option_lists):
        permutation_dict = dict(zip(keys, combination))
        total_cost = sum(cost for _, cost in combination)
        all_permutations.append((permutation_dict, total_cost))
    return all_permutations


get_permutations_of_options_with_costs(vertex_options_and_costs)